# AVSS: inference demo



## 1. Clone the repository
While the repo is private, add a GitHub token to Colab Secrets under the name `GITHUB_TOKEN`. For a public repo nothing is needed

In [1]:
import os, subprocess

REPO = "github.com/YohanKIM-creator/AVSS.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@{REPO}"
except Exception:
    url = f"https://{REPO}"

if not os.path.exists("/content/AVSS"):
    _ = subprocess.run(["git", "clone", "-q", url, "/content/AVSS"], check=True)
%cd /content/AVSS

/content/AVSS


## 2. Install dependencies
torch / torchaudio / numpy are already in Colab, so they are skipped to avoid version conflicts

In [9]:
!grep -v -E "^(torch|torchvision|torchaudio|numpy)([<>=]|$)" requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt

## 3. Lipreading frontend code and weights

In [3]:
!git clone -q https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git /content/lipreading
!python scripts/download_checkpoint.py
!ls -la saved/final saved/lipreading

saved/final:
total 41412
drwxr-xr-x 2 root root     4096 Nov 26 17:42 .
drwxr-xr-x 4 root root     4096 Nov 26 17:42 ..
-rw-r--r-- 1 root root 42390465 Nov 26 14:22 model_best.pth

saved/lipreading:
total 142232
drwxr-xr-x 2 root root      4096 Nov 26 17:42 .
drwxr-xr-x 4 root root      4096 Nov 26 17:42 ..
-rw-r--r-- 1 root root 145630317 Sep  3  2022 lrw_resnet18_mstcn_video.pth


## 4. Data

Expected structure:
```
<DATA_DIR>/
  audio/mix/*.wav
  audio/s1/*.wav   # optional
  audio/s2/*.wav   # optional
  mouths/*.npz
```
Either upload a zip with this structure (cell below), or set `DATA_DIR` to a folder on a mounted Google Drive

In [4]:
DATA_DIR = "/content/data"

from google.colab import files
uploaded = files.upload() # choose a .zip with audio/ and mouths/ inside
zip_name = next(iter(uploaded))
!mkdir -p {DATA_DIR} && unzip -q -o "{zip_name}" -d {DATA_DIR}
!ls {DATA_DIR} {DATA_DIR}/audio

Saving custom_test.zip to custom_test.zip
/content/data:
audio  mouths

/content/data/audio:
mix  s1  s2


Alternative: Google Drive (skip the upload cell above).

In [5]:
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/<your_folder>"

## 5. Inference
Predictions are saved to `saved/inference/s1/<mix_name>.wav` and `saved/inference/s2/<mix_name>.wav`.

In [6]:
SAVE_DIR = "/content/AVSS/saved/inference"
!python inference.py datasets.test.data_dir={DATA_DIR} inferencer.lip_repo=/content/lipreading inferencer.save_path={SAVE_DIR}

Saved 4 embeddings to /content/AVSS/saved/inference/lip_emb
3438853
Loading model weights from: saved/final/model_best.pth ...
Missing keys: []
 Unexpected keys: []
test: 100% 3/3 [00:15<00:00,  5.11s/it]
    test_SISNRi    : 10.840528806050619


## 6. Metrics (only if `audio/s1` and `audio/s2` exist)

In [10]:
!python -m scripts.calc_metrics --pred-dir {SAVE_DIR} --data-dir {DATA_DIR}

Mean SI_SNRi = 10.840532620747885
Mean SDRi = 11.119255383809408
Mean PESQ = 2.133713642756144
Mean STOI = 0.9132106549015052


## 7. Listen to an example

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

name = sorted(Path(SAVE_DIR, "s1").glob("*.wav"))[0].name
print(name)
for title, path in [("mix", Path(DATA_DIR, "audio", "mix", name)),
                    ("estimate s1", Path(SAVE_DIR, "s1", name)),
                    ("estimate s2", Path(SAVE_DIR, "s2", name))]:
    print(title)
    display(Audio(str(path)))